<a href="https://colab.research.google.com/github/TCU-DCDA/WRIT20833_2026/blob/main/notebooks/codeAlongs/WRIT20833_Scraped_Data_Cleaner_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WRIT 20833 — Clean Your Scraped Data

**When Coding Meets Culture: Developing Data-Driven Opinions**

Make an editable copy of this notebook by going to **File > Save a copy in Drive**

**New to Colab, or submitting?** See [Working in Colab](https://tcu-dcda.github.io/WRIT20833_2026/guides/working-in-colab.html).

This notebook is for you if you collected your data with a **no-code scraper** (Instant Data Scraper,
Easy Scraper, or a similar browser extension) and downloaded a CSV or Excel file. It turns that raw
download into a clean table with **one text column**, saved as a CSV you'll bring to HW3 (sentiment)
and HW4 (topics), and maybe to your capstone.

**Used the YouTube Comment Collector?** You don't need this notebook. The collector already cleaned
your comments (its Step 8) and saved a CSV with a `comment_clean` column. Go straight to HW3.

**How this notebook works.** The code cells come in three kinds:

- **Edit this one.** Your choices about *your* data go here. There are three of these cells.
- **Read this one.** It uses things you already know: lists, loops, functions, pandas. Read it and
  make sure you could explain it.
- **Plumbing.** It loads files or checks for mistakes. Run it; you don't need to read every line.

Unlike the collector, you'll probably run some cells **more than once**: look at your data, make a
choice, run again, look again. That back-and-forth *is* cleaning. If a cell gives you an error, read
the message: most of them say exactly what to fix.

## Why we're doing this

- **From Pandas 02:** cleaning isn't neutral housekeeping. Every fix is a choice, and every choice
  changes what the data can say.
- **Where you'll use it:** the file you save at the end is your dataset for HW3 and HW4, and the
  starting point for your capstone if you keep working with it.
- **The question to keep asking:** what does this flatten, who chose it, and can you trust it? A
  scraper made choices *before* you did. It decided what counts as a "row" and what counts as a
  "column," and it didn't know which part of the page you cared about.

# Before You Clean: What Did You Collect?

A scraper reads a page the way your browser shows it. So it grabs whatever was on screen: the posts
or reviews or comments you wanted, but also usernames, profile links, "Reply" buttons, timestamps like
"3d," and anything else that sat in the same box on the page.

- **Scrapers collect people, not only text.** Your file probably has usernames, display names, or
  links to profiles. You don't need them to study what was said, so in Step 4 you'll leave them out.
  Keeping them would turn your dataset into a list of people who never agreed to be in it.
- **You got what loaded, not what exists.** Infinite scroll, "load more" buttons, the site's own
  ranking, and whether you were logged in all shaped what reached your screen. That's your sample,
  and your writeup should say how you got it.
- **Long texts may be cut off.** If the page showed "…See more," the scraper usually saw only the
  shortened version. Cleaning can't bring back words that were never collected. If most of your
  texts end in "…," it may be worth scraping again after expanding them.

## Check Two Rules, Not One

Pandas 01 showed you how to read a site's `robots.txt`. That file is the site's rule for **programs**.
The **terms of use** (usually linked at the bottom of every page) are the site's rule for **you**, and
the two can disagree.

Rotten Tomatoes is a real example. Its `robots.txt` doesn't block its review pages, so a robots-only
check says "go ahead." Its terms of use say the opposite: no "robots, spiders, crawlers, devices,
scripts or other data gathering or extraction software" to "scrape, data mine, or aggregate any
content." A browser-extension scraper is that kind of software.

**What to do:**

1. **Check both.** Open the terms and search the page (Ctrl+F or ⌘F) for *scrape*, *robot*,
   *automated*, and *data mining*.
2. **If the terms allow it, or the source is built for collecting** (like the YouTube API in the
   Comment Collector, or a published dataset), go ahead. That's the better path when one exists.
3. **If the terms forbid it**, and you have a real reason to study *this* site, you may go ahead for
   this class on four conditions:
   - **Small:** one scrape of what you could read yourself by scrolling, not a tool crawling page
     after page.
   - **Private:** the raw file stays in your own Drive and your submissions. No public GitHub, no
     shared links.
   - **No people:** leave out usernames and handles (Step 4 shows how).
   - **Said out loud:** your methods section names the site's rule and explains why you went ahead.
4. **Not sure what the terms mean?** Ask before you build on the data.

This is the course's question in a new place: **who wrote these rules, for whom, and does "public"
mean "fair game"?** Record what you found in Step 1.

# Step 1: Record How You Collected

Write down how you got your data **now**, while you remember. This record is part of your method:
"I scraped the first 200 reviews sorted by Most Helpful, logged out, on November 13" tells a reader
what your sample can and can't show. `SITE_TERMS` is what you found when you checked the site's terms
of use (just above). In Colab these lines also appear as a small form to the right of the code.

In [ ]:
# EDIT THIS ONE: YOUR RECORD: how you collected this data
SOURCE_URL = ""  # @param {type:"string"}
COLLECTED_ON = ""  # @param {type:"string"}
SCRAPER = "Instant Data Scraper"  # @param ["Instant Data Scraper", "Easy Scraper", "Other"]
HOW_I_COLLECTED = ""  # @param {type:"string"}
SITE_TERMS = "Didn't check yet"  # @param ["Allow scraping", "Forbid scraping", "Don't mention it", "Didn't check yet"]

# Examples:
# SOURCE_URL = "https://www.example.com/forum/thread/123"
# COLLECTED_ON = "2026-11-13"
# HOW_I_COLLECTED = "sorted by Top, clicked 'load more' 4 times, logged out"

# Step 2: Upload and Load Your File

Run the first cell below. A **Choose Files** button appears under it: click it and pick the CSV or
Excel file your scraper downloaded. The cell waits until the upload finishes, then remembers the
file's name for you. Then run the second cell, which reads the file into a table.

**Colab forgets uploaded files** when your session ends. If you come back later, run the upload
cell again.

**No button?** Some browsers block it. Upload with the **Files** icon (a folder) in the left sidebar instead,
then in the first cell replace everything below the comment with one line naming your file, like
`MY_FILE = "reviews.csv"`.

In [ ]:
# PLUMBING: click "Choose Files" and pick the file your scraper downloaded
from google.colab import files

uploaded = files.upload()          # waits for you to choose a file; returns {file name: contents}
if len(uploaded) == 0:
    raise ValueError("No file was chosen. Run this cell again and pick your scraper's CSV or Excel file.")

MY_FILE = list(uploaded.keys())[0]
if len(uploaded) > 1:
    print(f"Warning: You chose {len(uploaded)} files; this notebook uses only the first one.")
print("Uploaded:", MY_FILE)

In [ ]:
# PLUMBING: load the tools, then read your file into a table called raw_df
import json
import os
from datetime import datetime, timezone

import pandas as pd

MY_FILE = MY_FILE.strip()
if MY_FILE == "":
    raise ValueError("MY_FILE is empty. Run the upload cell above and choose your file.")

if not os.path.exists(MY_FILE):
    files_here = []
    for name in sorted(os.listdir(".")):
        if name.lower().endswith((".csv", ".xlsx")):
            files_here.append(name)
    raise FileNotFoundError(f"Can't find a file named {MY_FILE!r}. CSV and Excel files uploaded so far: "
                            f"{files_here or 'none'}. Check the spelling, or upload the file again.")

# dtype=str reads every cell as text, and keep_default_na=False keeps a post that says "NA" or
# "None" as words instead of turning it into a missing value
if MY_FILE.lower().endswith(".xlsx"):
    raw_df = pd.read_excel(MY_FILE, dtype=str, keep_default_na=False)
elif MY_FILE.lower().endswith(".csv"):
    try:
        raw_df = pd.read_csv(MY_FILE, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    except UnicodeDecodeError:
        raw_df = pd.read_csv(MY_FILE, dtype=str, keep_default_na=False, encoding="latin-1")
else:
    raise ValueError("This notebook reads .csv and .xlsx files. Download your data from the scraper "
                     "as CSV (or re-save it as CSV), upload it, and try again.")

raw_df = raw_df.fillna("")      # any cell that is still blank becomes an empty string
print(f"Loaded {MY_FILE}: {len(raw_df)} rows, {len(raw_df.columns)} columns.")

if "comment_clean" in raw_df.columns and "comment_id" in raw_df.columns:
    print()
    print("Note: This looks like a file from the YouTube Comment Collector, which already cleaned it.")
    print("   You don't need this notebook: take the file straight to HW3 with")
    print(f'   MY_CSV = "{MY_FILE}"  and  MY_TEXT_COLUMN = "comment_clean"')

# Step 3: Diagnose: What Did the Scraper Give You?

Same rule as Pandas 02: **look before you touch anything.** Start with the first few rows.

In [ ]:
# READ THIS ONE: the first five rows, every column
raw_df.head()

Scrapers name columns after the page's hidden code, so you may see names like `css-1qaijid`,
`reviews-cards (3)`, or `href`. Those names tell you nothing; what's *in* each column tells you everything. The
next cell prints one short report per column:

- **filled in**: how many rows have something in this column
- **different values**: how many distinct values it holds. Post text is almost all different;
  usernames, dates, and button labels repeat a lot
- **characters each**: the average length. Your text is usually the column with the longest values
- **example**: the first non-empty value, cut to 70 characters

In [ ]:
# READ THIS ONE: a short report on every column, to find the one that holds your text
for column_name in raw_df.columns:
    values = raw_df[column_name].str.strip()
    filled = values[values != ""]                     # keep only the non-empty cells

    if len(filled) > 0:
        example = " ".join(filled.tolist()[0].split())[:70]   # line breaks shown as spaces
        average_length = round(filled.str.len().mean())   # .str.len() is each value's length
    else:
        example = "(this column is empty)"
        average_length = 0

    print(f'"{column_name}"')
    print(f"    filled in {len(filled)} of {len(raw_df)} rows | {len(filled.unique())} different values "
          f"| about {average_length} characters each")
    print(f"    example: {example}")
    print()

**Read the report and decide:**
- Which column (or columns) hold the **text** you want to study? Long, almost all different, words
  rather than links.
- Which hold **useful context** you might analyze later: a date, a rating, a like count, a title?
- Which hold **people**: usernames, display names, profile links, profile pictures? Leave those out.
- Which are **noise**: links to images, button labels, columns that are nearly empty?
- Is there a column that only ever says **"See More"** or **"Read more"**? That's the page's button,
  and it marks the rows whose text was **cut short** on screen, so the scraper only got the beginning.
  Count them: that's how many of your texts are incomplete. Leave the column out, but say so in your
  writeup (or scrape again after expanding them, if the site lets you).

Not sure? Look at a few rows of one column. Change the name in quotes below to any column name
from the report.

In [ ]:
# READ THIS ONE: peek at ten values from one column (change the name to any column from the report)
column_to_peek = raw_df.columns[0]      # or type a name, like: column_to_peek = "reviews-cards (3)"
raw_df[column_to_peek].head(10)

**Repeated rows.** As you scroll or click "next," pages often overlap, and the scraper reads the same
post twice. `.duplicated()` from Pandas 02 finds rows where **every** column matches.

In [ ]:
# READ THIS ONE: how many rows are exact copies of an earlier row?
print(raw_df.duplicated().sum(), "rows are exact repeats of an earlier row (every column matches)")

# Step 4: Choose Your Columns

Copy the column names from Step 3's report into two lists. Copy them **exactly**: capitals and
spaces count.

- **`TEXT_COLUMNS`**: the column that holds your text, in a list: `["reviews-cards (3)"]`. Usually that's
  one column. If the scraper split each post into pieces (a title column and a body column, say),
  list them all in reading order and they'll be glued together, with a space between.
- **`KEEP_COLUMNS`**: other columns worth keeping for later, like a date, rating, or like count. It
  can stay empty: `[]`. **Leave out anything that names or links to a person.**
- **`DROP_REPEATED_ROWS`**: `True` removes the exact repeats Step 3 counted. Leave it `True` unless
  your file has *only* a text column. In that case two different people who wrote the same words
  ("This!") would look like a repeat, so set it to `False`.
- **`NEW_NAMES`** (optional): better names for the columns you kept, so `reviews-cards (3)` can
  become `review`. Old name on the left, new name on the right: `{"reviews-cards (3)": "review"}`. It can stay
  empty: `{}`. How renaming works is explained a little further down, before the cell that does it.

Add a `#comment` saying **why** you chose these columns. That's part of your method too.

In [ ]:
# EDIT THIS ONE: YOUR COLUMNS: copy the names from Step 3's report
TEXT_COLUMNS = []          # e.g. ["reviews-cards (3)"]  or  ["title", "body"]
KEEP_COLUMNS = []          # e.g. ["reviews-cards (4)"]  (nothing that names or links to a person)
DROP_REPEATED_ROWS = True
NEW_NAMES = {}             # e.g. {"reviews-cards (3)": "review", "reviews-cards (4)": "rating"}

# Why I chose these columns (and these names):
#

In [ ]:
# PLUMBING: check that every name you typed is really a column in your file
if len(TEXT_COLUMNS) == 0:
    raise ValueError("TEXT_COLUMNS is empty. Put the name of your text column inside the brackets, "
                     'in quotes, like ["reviews-cards (3)"], then run that cell and this one again.')
for column_name in TEXT_COLUMNS + KEEP_COLUMNS:
    if column_name not in raw_df.columns:
        raise KeyError(f"No column named {column_name!r}. Copy the name exactly from Step 3 (capitals and "
                       f"spaces count). Your columns are: {list(raw_df.columns)}")
for column_name in TEXT_COLUMNS:
    if column_name in KEEP_COLUMNS:
        raise ValueError(f"{column_name!r} is in both lists. Put it in TEXT_COLUMNS only.")
if "text_clean" in KEEP_COLUMNS:
    raise ValueError('Leave "text_clean" out of KEEP_COLUMNS: the next cell makes a new one.')

# NEW_NAMES: each old name must be a column you chose; new names must be new and different
final_names = []
for column_name in TEXT_COLUMNS + KEEP_COLUMNS:
    final_names.append(NEW_NAMES.get(column_name, column_name))   # the name it will end up with
for old_name in NEW_NAMES:
    if old_name not in TEXT_COLUMNS + KEEP_COLUMNS:
        raise KeyError(f"NEW_NAMES renames {old_name!r}, but that isn't in TEXT_COLUMNS or KEEP_COLUMNS. "
                       "You can only rename a column you chose to keep.")
    if str(NEW_NAMES[old_name]).strip() == "":
        raise ValueError(f"NEW_NAMES gives {old_name!r} an empty new name. Type a name inside the quotes.")
for new_name in final_names:
    if new_name == "text_clean":
        raise ValueError('"text_clean" is taken: the next cell makes that column. Pick another name.')
    if final_names.count(new_name) > 1:
        raise ValueError(f"Two of your columns would both be called {new_name!r}. Give each a different name.")
print("Column names check out.")

The next cell builds your working table in four moves:

1. **Drop exact repeats on the full table**, before any column is removed. This order matters. While
   the username, link, and time are still there, a row only counts as a repeat if *all* of them
   match: the same post, read twice. Once those columns are gone, two different people who wrote
   "Same." would look identical.
2. **Keep only the columns you chose**, as a copy (Pandas 02's rule: never change the raw data).
3. **Make one text column, `text_clean`.** Glue the text columns together, then collapse runs of
   spaces, tabs, and line breaks into single spaces, like the collector's `normalize_whitespace`. The
   columns you listed stay in the table, untouched, so you can always compare.
4. **Rename** the columns listed in `NEW_NAMES`.

**Renaming columns.** A scraper's names describe the web page's code, not your data. Naming a column
is a small act of interpretation: calling `reviews-cards (4)` "rating" is a claim about what it holds, so check
the values first. Clear names also make the file readable to the next person who opens it (including
you, at capstone time).

**The structure:**
```python
my_df = my_df.rename(columns={"old_name": "new_name", "other_old": "other_new"})
```
- The dictionary has the same shape as Pandas 02's `.replace({...})`: old on the left, new on the right.
  `.replace()` changes **values** in the cells; `.rename(columns=...)` changes the **column names**.
- Columns that aren't in the dictionary keep their names.
- Like most pandas methods, it hands back a new table, so save it with `my_df = ...`.

Gluing columns is new. You've seen `+` join two strings (`"Fort" + " " + "Worth"`). It does the same
for two **columns**, row by row.

**The structure:**
```python
my_df["new_column"] = my_df["first_column"] + " " + my_df["second_column"]
```
- Row by row, the value in `first_column`, a space, and the value in `second_column` become one string.
- Both columns must hold text. Ours do, because Step 2 read every cell as text.
- The loop below does this once per column in `TEXT_COLUMNS`, so one column or three both work.

In [ ]:
# READ THIS ONE: drop repeats, keep your columns, build one clean text column, rename
def normalize_whitespace(text):
    # .split() with no argument splits on ANY run of spaces, tabs, or line breaks;
    # " ".join() glues the pieces back together with single spaces
    return " ".join(str(text).split())


# 1. Exact repeats, checked while every column is still there
repeats_removed = 0
working_df = raw_df
if DROP_REPEATED_ROWS:
    repeats_removed = raw_df.duplicated().sum()
    working_df = raw_df.drop_duplicates().reset_index(drop=True)

# 2. Only the columns you chose, as a copy
chosen_df = working_df[TEXT_COLUMNS + KEEP_COLUMNS].copy()

# 3. One text column: glue, then tidy the whitespace
chosen_df["text_clean"] = ""
for column_name in TEXT_COLUMNS:
    chosen_df["text_clean"] = chosen_df["text_clean"] + " " + chosen_df[column_name]
chosen_df["text_clean"] = chosen_df["text_clean"].apply(normalize_whitespace)   # run it on every row

# Put text_clean first, so it's the first thing you see
chosen_df = chosen_df[["text_clean"] + TEXT_COLUMNS + KEEP_COLUMNS]

# 4. Your names instead of the scraper's (an empty NEW_NAMES changes nothing)
chosen_df = chosen_df.rename(columns=NEW_NAMES)

print("Rows in the raw file:        ", len(raw_df))
print("Exact repeats removed:       ", repeats_removed)
print("Rows in your working table:  ", len(chosen_df))
chosen_df.head()

# Step 5: Look for Leftovers

Scrapers pick up bits of the page that aren't anyone's words. They come in two kinds, and you'll
handle them differently in Step 6:

- **Whole-row junk:** a row whose entire text is a button or label, like `Reply`, `Sponsored`,
  `Load more`, or a blank row. These are usually very short, so the next cell lists every short text.
- **Junk inside real text:** a real post with `See more`, `Translate`, `(edited)`, or `Show
  replies` stuck to it. The random sample below is the best place to spot these.

Nothing is removed in this step. You're only looking.

In [ ]:
# READ THIS ONE: every text of three words or fewer, and how many times each appears
word_counts = chosen_df["text_clean"].str.split().str.len()     # words in each row
short_texts = chosen_df[word_counts <= 3]["text_clean"].value_counts()

print(len(short_texts), "different short texts (an empty row shows up as a blank line at the top)")
short_texts.head(25)

Short isn't the same as junk. "So true." and "❤️" are real responses from real people; keep them.
Only `Reply` and its cousins are the page talking.

In [ ]:
# READ THIS ONE: longer texts that appear more than once (shown, not removed)
long_texts = chosen_df[word_counts > 3]["text_clean"]
repeated_texts = long_texts.value_counts()
repeated_texts = repeated_texts[repeated_texts > 1]
print(len(repeated_texts), "longer texts appear more than once")
repeated_texts.head(10)

A long text that appears several times might be a spam bot, a copy-and-paste campaign, or a quote
everyone is reacting to. That's a judgment about *your* data, not an automatic step. If you decide
some are junk, you can list them in Step 6.

In [ ]:
# READ THIS ONE: eight rows chosen at random, printed in full (change random_state to see others)
sample = chosen_df.sample(min(8, len(chosen_df)), random_state=1)
for text in sample["text_clean"]:
    print("•", text)
    print()

# Step 6: Remove Leftovers

Fill in the two lists from what you saw in Step 5, run this cell and the next, then read Step 7's
sample. Change the lists and re-run as many times as you like: the next cell always starts again from
Step 4's table, so nothing gets removed twice.

- **`REMOVE_PHRASES`**: bits of page text to cut out of every row, like `"See more"`. A phrase is cut
  **wherever it appears, even in the middle of a real sentence**, so make each one as specific as you
  can. `"… See more"` is safe; `"more"` would damage every post that uses the word "more."
- **`JUNK_TEXTS`**: rows to drop entirely when their whole text is **exactly** this, like `"Reply"`.
  A post that merely *contains* "Reply" is left alone.

Both can stay empty: `[]`.

**What we deliberately *don't* clean:** capital letters, punctuation, emoji, "not," "lol," "/s."

| original | "aggressively cleaned" |
|---|---|
| `This is NOT okay!!! 😡` | `okay` |
| `Oh great, ANOTHER law. Just what we needed /s` | `oh great another law needed` |

*(Two invented examples.)* **VADER reads capitals, "!!!", emoji, and "not"** as signals, so HW3 needs
them. HW4's topic modeling does its own heavier cleaning inside that notebook, on a copy.

In [ ]:
# EDIT THIS ONE: YOUR LEFTOVERS: fill these in after looking at Step 5
REMOVE_PHRASES = []     # cut out of every row, e.g. ["… See more", "Translate"]
JUNK_TEXTS = []         # drop the row when its whole text is exactly this, e.g. ["Reply", "Sponsored"]

The next cell drops junk rows with a loop over `JUNK_TEXTS`. Each pass is the boolean filtering
from Pandas 01, with `!=` (not equal) instead of `>`.

**The structure:**
```python
my_df = my_df[my_df["column_name"] != "unwanted value"]
```
- `my_df["column_name"] != "unwanted value"` is `True` for every row that **isn't** the unwanted value.
- The outer `my_df[ ... ]` keeps only the `True` rows, so the unwanted ones are left out.
- Run it once per unwanted value (that's what the loop does), and each pass removes one more.

In [ ]:
# READ THIS ONE: apply your choices, starting fresh from Step 4's table every time
clean_df = chosen_df.copy()

# Cut each phrase out of every row, then tidy the gaps the cuts left
for phrase in REMOVE_PHRASES:
    clean_df["text_clean"] = clean_df["text_clean"].str.replace(phrase, "", regex=False)
clean_df["text_clean"] = clean_df["text_clean"].apply(normalize_whitespace)

# Drop rows that are exactly one of your junk texts
before = len(clean_df)
for junk in JUNK_TEXTS:
    clean_df = clean_df[clean_df["text_clean"] != junk]
junk_removed = before - len(clean_df)

# Drop rows with no text left at all
before = len(clean_df)
clean_df = clean_df[clean_df["text_clean"] != ""]
empty_removed = before - len(clean_df)

clean_df = clean_df.reset_index(drop=True)

print("Junk rows removed: ", junk_removed)
print("Empty rows removed:", empty_removed)
print("Rows left:         ", len(clean_df))
for junk in JUNK_TEXTS:
    if junk not in chosen_df["text_clean"].tolist():
        print(f"Warning: No row was exactly {junk!r}. Check the spelling and spaces against Step 5.")

# Step 7: Check Your Clean Table

Before you save, make sure the table is what you think it is. The question to ask of every row:
**is this one whole text by one person?** If the scraper split one post across several rows, or put
several posts in one row, your HW3 counts will mean something different, and your writeup should say so.

In [ ]:
# READ THIS ONE: from raw download to clean table, step by step
word_counts = clean_df["text_clean"].str.split().str.len()

print("Rows in the raw file:     ", len(raw_df))
print("  exact repeats removed:  ", repeats_removed)
print("  junk rows removed:      ", junk_removed)
print("  empty rows removed:     ", empty_removed)
print("Rows in the clean table:  ", len(clean_df))
if len(clean_df) > 0:
    print()
    print("Average length:           ", round(word_counts.mean(), 1), "words")
    print("Shortest / longest:       ", word_counts.min(), "/", word_counts.max(), "words")

In [ ]:
# READ THIS ONE: five clean rows at random
sample = clean_df.sample(min(5, len(clean_df)), random_state=2)
for text in sample["text_clean"]:
    print("•", text[:300])
    print()

Something look wrong? Go back: Step 4 to change your columns, Step 6 to change your leftovers, then
run every cell from there down again.

Last, the record from Step 1 plus what you did here, as a dictionary you can save beside your CSV.

In [ ]:
# READ THIS ONE: a record of how this dataset was made and cleaned
collection_info = {
    "source_url": SOURCE_URL,
    "collected_on": COLLECTED_ON,
    "scraper": SCRAPER,
    "how_i_collected": HOW_I_COLLECTED,
    "site_terms_on_scraping": SITE_TERMS,
    "raw_file": MY_FILE,
    "cleaned_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "rows_in_raw_file": len(raw_df),
    "text_columns": TEXT_COLUMNS,
    "kept_columns": KEEP_COLUMNS,
    "renamed_columns": NEW_NAMES,
    "exact_repeats_removed": int(repeats_removed),
    "phrases_removed": REMOVE_PHRASES,
    "junk_texts_removed": JUNK_TEXTS,
    "junk_rows_removed": junk_removed,
    "empty_rows_removed": empty_removed,
    "rows_in_clean_table": len(clean_df),
}
collection_info

# Step 8: Save Your Dataset

This saves `clean_df` as a CSV named after your file, with `_clean` added, then reloads it **the same
way HW3 and HW4 will**, to check that nothing got lost on the way.

**Colab forgets files** when your session ends, so download the CSV (or copy it to your Google
Drive) before you close the tab.

**If you open the CSV in Excel** and see clutter like `‚ÄúSo kids‚Äù` where curly quotes should be,
Excel guessed the wrong encoding. The file is fine; Python and Google Sheets read it correctly. If you
still see clutter, import it through Excel's **Data → From Text/CSV** and choose UTF-8.

In [ ]:
# READ THIS ONE: save the CSV, then reload it exactly as HW3's setup cell does
file_stem = os.path.splitext(os.path.basename(MY_FILE))[0]      # the name without .csv or .xlsx
csv_filename = file_stem + "_clean.csv"
# "utf-8-sig" is UTF-8 plus an invisible marker, so Excel reads emoji and curly quotes correctly
clean_df.to_csv(csv_filename, index=False, encoding="utf-8-sig")

check = pd.read_csv(csv_filename, keep_default_na=False)       # HW3/HW4 read it this way
same_rows = len(check) == len(clean_df)
same_text = list(check["text_clean"].astype(str)) == list(clean_df["text_clean"])
print("Saved", csv_filename)
print("Reloads with the same rows:", same_rows, "| same text, emoji included:", same_text)
print()
print("For HW3 and HW4, copy these two lines into the setup cell:")
print(f'    MY_CSV = "{csv_filename}"')
print('    MY_TEXT_COLUMN = "text_clean"')

In [ ]:
# PLUMBING: download the CSV to your computer
from google.colab import files
files.download(csv_filename)

**Optional:** download the record from Step 7 as a JSON file too. Keep it beside your CSV; it holds
the details your methods paragraph needs.

In [ ]:
# PLUMBING (optional): save and download the collection record
info_filename = file_stem + "_clean_info.json"
with open(info_filename, "w", encoding="utf-8") as f:
    json.dump(collection_info, f, indent=2, ensure_ascii=False)
files.download(info_filename)

# Where This Goes Next

**In HW3 (sentiment) and HW4 (topics),** upload your `_clean.csv` to Colab (Files panel → upload)
and paste the two lines Step 8 printed into the "Choose your data" part of the setup cell:

```python
MY_CSV = "yourfile_clean.csv"
MY_TEXT_COLUMN = "text_clean"
```

The setup cell prints a `DATA SOURCE:` line. Check that it names your file before you go on.

**For your writeup, record:** the site and page, the date you collected, how you scrolled or paged,
which columns you kept and why, and what you removed. All of it is in `collection_info`. And keep
asking the question from the top of this notebook: what did the scraper decide for you, and whose
words did it miss?

# Playground

Some things to try:
- **Try another choice.** Put a different column in `TEXT_COLUMNS`, re-run from Step 4, and compare
  Step 7's numbers. What changes about what your dataset can say?
- **Find your topic.** Show only the rows that mention a word you care about:
  `clean_df[clean_df["text_clean"].str.lower().str.contains("your word")]`
- **Count a context column.** If you kept a date or rating column, try `.value_counts()` on it. Is
  your sample spread out, or bunched together?

In [ ]:
# use this space to experiment!